# Field Scan : expérience 1, comparaison de réseaux pour le classifieur de maladies

**Question** : un réseau plus puissant qu'EfficientNet-B0 généralise-t-il mieux et rejette-t-il mieux les maladies inconnues, en restant utilisable sur processeur ?

**Réseaux** (pré-entraînés sur ImageNet, torchvision) : EfficientNet-B0 (référence, résultats repris de l'entraînement précédent), EfficientNet-B2, ConvNeXt-Tiny.

**Même recette pour tous** (v2) : recadrages de 15 à 100 %, masquage aléatoire, 12 époques, époque choisie sur la moyenne des F1 de validation (images entières et recadrages centrés). Seule différence, fixée à l'avance : taux d'apprentissage 0,0001 pour ConvNeXt (pratique standard pour ce réseau), 0,0003 pour les EfficientNet.

**Mesures** : F1 de validation avec 4 vues (images entières et recadrées), fausses alertes sur feuilles saines ; inconnues simulées (trois modèles « deux classes », 6 époques) ; taille ONNX et temps par photo sur processeur (4 vues).

**Règle fixée avant les résultats** : un réseau remplace EfficientNet-B0 s'il remplit les trois conditions :
1. AUROC moyenne des inconnues simulées (probabilité maximale, la méthode retenue) supérieure à celle de B0 ;
2. F1 de validation (images entières, 4 vues) au plus 0,005 sous celui de B0 ;
3. au plus 100 ms par photo (4 vues) sur processeur.
Si plusieurs réseaux conviennent, celui qui a la meilleure AUROC.

**Aucun test n'est utilisé** (BananaLSD et la Cordana péruvienne ont déjà été consultés).

**Avant de lancer** : Input = le jeu `field-scan-feuilles` **et** la sortie du carnet d'entraînement (dossier `modeles_maladies_perou`, pour reprendre B0 sans le réentraîner) ; Accelerator = GPU ; Internet = activé ; puis **Save Version > Save & Run All**. Durée : environ 4 heures. Si l'exécution s'interrompt, attacher sa sortie en entrée et relancer : les modèles déjà entraînés sont relus.

## 1. Données et reprise

In [ ]:
import os, glob, zipfile, shutil
SORTIE = '/kaggle/working/comparaison_reseaux'        # conservé par Kaggle (onglet Output)
REPRENDRE = True                                         # relire les modèles déjà entraînés au lieu de les réentraîner
os.makedirs(SORTIE, exist_ok=True)

# Données : index.csv dans un jeu de données attaché ; sinon, le zip est décompressé dans /tmp (hors des résultats)
index_trouves = [p for p in glob.glob('/kaggle/input/**/index.csv', recursive=True) if os.path.isdir(os.path.join(os.path.dirname(p), 'images'))]
if index_trouves:
    DONNEES = os.path.dirname(index_trouves[0])
else:
    zips = glob.glob('/kaggle/input/**/feuilles_pret*.zip', recursive=True)
    assert zips, "Aucun index.csv ni feuilles_pret*.zip dans /kaggle/input : ajoute le jeu de données (panneau Input)"
    DONNEES = '/tmp/feuilles'
    if not os.path.exists(f'{DONNEES}/index.csv'):
        with zipfile.ZipFile(zips[0]) as z:          # « \\ » remplacé par « / » si le zip vient de PowerShell
            for m in z.infolist():
                chemin = m.filename.replace('\\', '/')
                if chemin.endswith('/'):
                    continue
                cible = os.path.join(DONNEES, chemin)
                os.makedirs(os.path.dirname(cible), exist_ok=True)
                with z.open(m) as src, open(cible, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
print('Données :', DONNEES)

# Reprise : résultats d'une exécution précédente attachés en entrée
# (B0 : sortie du carnet d'entraînement ; autres réseaux : sortie d'une exécution précédente de ce carnet)
for anciens in glob.glob('/kaggle/input/**/modeles_maladies_perou', recursive=True) + glob.glob('/kaggle/input/**/comparaison_reseaux', recursive=True):
    for f in os.listdir(anciens):
        if not os.path.exists(os.path.join(SORTIE, f)):
            shutil.copy(os.path.join(anciens, f), SORTIE)
    print('Reprise depuis', anciens, ':', sorted(os.listdir(SORTIE)))

import pandas as pd
index = pd.read_csv(f'{DONNEES}/index.csv')
index = index[index.retenue].reset_index(drop=True)
assert all(os.path.exists(f'{DONNEES}/{c}') for c in index.chemin.sample(200, random_state=0)), "images manquantes"
print(index.groupby(['ensemble', 'classe']).size().unstack(fill_value=0))

## 2. Versions

In [ ]:
import torch, torchvision, platform, random, time, json, numpy as np
GRAINE = 2026
def graine():
    random.seed(GRAINE); np.random.seed(GRAINE); torch.manual_seed(GRAINE)
graine()
VERSIONS = dict(torch=torch.__version__, torchvision=torchvision.__version__, python=platform.python_version(),
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun')
print(VERSIONS)
assert torch.cuda.is_available(), "Pas de GPU : Settings > Accelerator > GPU T4 x2 ou GPU P100"
DEVICE = 'cuda'

## 3. Jeux de données et augmentations

In [ ]:
import io
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

CLASSES = ['cercosporiose_noire', 'fusariose', 'saine']
MOYENNE, ECART = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
NORMALISATION = T.Normalize(MOYENNE, ECART)
moy, ec = torch.tensor(MOYENNE)[:, None, None], torch.tensor(ECART)[:, None, None]   # pour réafficher les images

class JPEGAleatoire:
    '''Recompression JPEG de qualité aléatoire, pour imiter des appareils différents.'''
    def __init__(self, p=0.5, qualites=(30, 95)): self.p, self.q = p, qualites
    def __call__(self, im):
        if random.random() > self.p: return im
        b = io.BytesIO(); im.save(b, 'JPEG', quality=random.randint(*self.q)); b.seek(0)
        return Image.open(b).convert('RGB')

class RecadrageCentre:
    def __init__(self, part): self.part = part
    def __call__(self, im):
        l, h = im.size; cl, ch = int(l * self.part), int(h * self.part)
        return im.crop(((l - cl) // 2, (h - ch) // 2, (l + cl) // 2, (h + ch) // 2))

def augmentations(recadrage_min, masquage):
    etapes = [JPEGAleatoire(), T.RandomResizedCrop(224, scale=(recadrage_min, 1.0)), T.RandomHorizontalFlip(),
              T.RandomVerticalFlip(), T.ColorJitter(0.3, 0.3, 0.3, 0.03),
              T.RandomApply([T.GaussianBlur(5, (0.1, 1.5))], p=0.2), T.ToTensor(), NORMALISATION]
    if masquage: etapes.append(T.RandomErasing(p=0.25, scale=(0.02, 0.15)))
    return T.Compose(etapes)

T_V1, T_V2 = augmentations(0.3, False), augmentations(0.15, True)
T_ENTIERE = T.Compose([T.Resize(224), T.CenterCrop(224), T.ToTensor(), NORMALISATION])
T_CENTRE = T.Compose([RecadrageCentre(0.6), T.Resize(224), T.CenterCrop(224), T.ToTensor(), NORMALISATION])

class Feuilles(Dataset):
    '''Images de l'index ; étiquette = rang dans `classes`, ou -1 pour une classe inconnue.'''
    def __init__(self, lignes, transformation, classes=CLASSES):
        self.lignes, self.t, self.classes = lignes.reset_index(drop=True), transformation, classes
    def __len__(self): return len(self.lignes)
    def __getitem__(self, i):
        r = self.lignes.iloc[i]
        y = self.classes.index(r.classe) if r.classe in self.classes else -1
        return self.t(Image.open(f'{DONNEES}/{r.chemin}').convert('RGB')), y

def lignes(ensemble, classes=None):
    l = index[index.ensemble == ensemble]
    return l if classes is None else l[l.classe.isin(classes)]

def chargeur(lignes_, transformation, classes=CLASSES, melanger=False, taille=64):
    return DataLoader(Feuilles(lignes_, transformation, classes), batch_size=taille, shuffle=melanger,
                      num_workers=2, pin_memory=True, drop_last=melanger)

L_VAL = chargeur(lignes('validation'), T_ENTIERE)
L_VAL_CENTRE = chargeur(lignes('validation'), T_CENTRE)
print(len(lignes('entrainement')), 'images d’entraînement |', len(L_VAL.dataset), 'de validation')

## 4. Réseaux, prédiction, mesures, entraînement

In [ ]:
import torch.nn as nn
import torchvision.models as tvm
from sklearn.metrics import f1_score, confusion_matrix

ARCHITECTURES = {
    'efficientnet_b0': dict(fabrique=tvm.efficientnet_b0, poids=tvm.EfficientNet_B0_Weights.DEFAULT, lr=3e-4),
    'efficientnet_b2': dict(fabrique=tvm.efficientnet_b2, poids=tvm.EfficientNet_B2_Weights.DEFAULT, lr=3e-4),
    'convnext_tiny':   dict(fabrique=tvm.convnext_tiny,   poids=tvm.ConvNeXt_Tiny_Weights.DEFAULT,   lr=1e-4),
}

def nom_fichier(arch, simu=None):
    '''B0 reprend les fichiers du carnet d'entraînement (meilleur_v2.pt, meilleur_simu_sans_*.pt).'''
    if arch == 'efficientnet_b0':
        return 'v2' if simu is None else f'simu_sans_{simu}'
    return arch if simu is None else f'{arch}_simu_sans_{simu}'

def creer_modele(arch, n_classes=len(CLASSES), pre_entraine=True):
    a = ARCHITECTURES[arch]
    m = a['fabrique'](weights=a['poids'] if pre_entraine else None)
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n_classes)
    return m

def caracteristiques_et_logits(m, x):
    h = m.avgpool(m.features(x))
    if isinstance(m, tvm.ConvNeXt):                 # tête : LayerNorm2d, aplatissement, couche linéaire
        f = m.classifier[1](m.classifier[0](h))
        return f, m.classifier[2](f)
    f = torch.flatten(h, 1)
    return f, m.classifier(f)

@torch.no_grad()
def predire(m, chargeur_):
    m.eval(); F, L, Y = [], [], []
    for x, y in chargeur_:
        with torch.autocast('cuda', dtype=torch.float16):
            f, l = caracteristiques_et_logits(m, x.to(DEVICE, non_blocking=True))
        F.append(f.float().cpu()); L.append(l.float().cpu()); Y.append(y)
    return torch.cat(F), torch.cat(L), torch.cat(Y)

VUES = {'identite': lambda x: x, 'miroir_horizontal': lambda x: torch.flip(x, [3]),
        'miroir_vertical': lambda x: torch.flip(x, [2]), 'rotation_180': lambda x: torch.flip(x, [2, 3])}

@torch.no_grad()
def logits_4vues(m, chargeur_):
    m.eval(); L, Y = [], []
    for x, y in chargeur_:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16):
            L.append(torch.stack([caracteristiques_et_logits(m, f(x))[1].float() for f in VUES.values()]).mean(0).cpu())
        Y.append(y)
    return torch.cat(L), torch.cat(Y)

def f1(L, Y):
    return round(float(f1_score(Y.numpy(), L.argmax(1).numpy(), average='macro')), 4)

def bilan(L, Y):
    m_ = confusion_matrix(Y.numpy(), L.argmax(1).numpy(), labels=range(len(CLASSES)))
    d = {f'rappel_{c}': round(float(m_[k, k] / m_[k].sum()), 3) for k, c in enumerate(CLASSES)}
    s = CLASSES.index('saine')
    d['fausses_alertes_saines'] = round(float((m_[s].sum() - m_[s, s]) / m_[s].sum()), 3)
    d['f1_macro'] = f1(L, Y)
    return d

PERTE = nn.CrossEntropyLoss(label_smoothing=0.1)
RECETTE = dict(epoques=12, lot=64, decroissance_poids=1e-4, lissage_etiquettes=0.1, programme='cosinus')

def entrainer(nom, arch, transformation, critere, classes=CLASSES, epoques=RECETTE['epoques']):
    '''critere : 'moyenne' (moyenne des F1 entières / recadrées) ou None (dernière époque : inconnues simulées).'''
    chemin, chemin_h = f'{SORTIE}/meilleur_{nom}.pt', f'{SORTIE}/historique_{nom}.json'
    m = creer_modele(arch, len(classes)).to(DEVICE)
    if REPRENDRE and os.path.exists(chemin) and os.path.exists(chemin_h):
        m.load_state_dict(torch.load(chemin)); h = json.load(open(chemin_h))
        print(f'{nom} : repris d’une exécution précédente (époque retenue {h["epoque_retenue"]})'); return m, h
    graine()
    L_tr = chargeur(lignes('entrainement', classes), transformation, classes, melanger=True)
    opt = torch.optim.AdamW(m.parameters(), lr=ARCHITECTURES[arch]['lr'], weight_decay=RECETTE['decroissance_poids'])
    prog = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epoques * len(L_tr))
    ech = torch.amp.GradScaler('cuda'); historique, meilleur = [], (-1, None)
    for ep in range(1, epoques + 1):
        m.train(); debut, total, n = time.time(), 0.0, 0
        for x, y in L_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                perte = PERTE(m(x), y)
            ech.scale(perte).backward(); ech.step(opt); ech.update(); prog.step()
            total += perte.item() * len(y); n += len(y)
        ligne = dict(epoque=ep, perte=round(total / n, 4))
        if critere is not None:
            _, Le, Ye = predire(m, L_VAL); _, Lr, Yr = predire(m, L_VAL_CENTRE)
            ligne.update(f1_entieres=f1(Le, Ye), f1_recadrages=f1(Lr, Yr))
            ligne['critere'] = round((ligne['f1_entieres'] + ligne['f1_recadrages']) / 2, 4)
        ligne['duree_s'] = round(time.time() - debut); historique.append(ligne); print(nom, ligne)
        valeur = ligne.get('critere', ep)
        if valeur > meilleur[0]:
            meilleur = (valeur, ep); torch.save(m.state_dict(), chemin)
    h = dict(historique=historique, epoque_retenue=meilleur[1], critere=critere, architecture=arch, lr=ARCHITECTURES[arch]['lr'])
    json.dump(h, open(chemin_h, 'w'), indent=2)
    m.load_state_dict(torch.load(chemin))
    print(f'{nom} : époque retenue {meilleur[1]}'); return m, h

## 5. Inconnues simulées et export (fonctions)

In [ ]:
!pip -q install onnx onnxruntime onnxscript
from sklearn.metrics import roc_auc_score
import onnxruntime as ort
ACCEPTATION_CIBLE = 0.95

def inconnues_simulees(arch):
    '''Pour chaque classe : modèle entraîné sur les deux autres (6 époques), la classe écartée (validation) joue l'inconnue.'''
    lignes_ = []
    for inconnue in CLASSES:
        connues = [c for c in CLASSES if c != inconnue]
        m, _ = entrainer(nom_fichier(arch, inconnue), arch, T_V2, None, classes=connues, epoques=6)
        _, Lc_, _ = predire(m, chargeur(lignes('validation', connues), T_ENTIERE, connues))
        _, Li, _ = predire(m, chargeur(lignes('validation', [inconnue]), T_ENTIERE, connues))
        for meth, fn in (('proba_max', lambda L: torch.softmax(L, 1).max(1).values), ('energie', lambda L: torch.logsumexp(L, 1))):
            sc, si = fn(Lc_), fn(Li)
            seuil = torch.quantile(sc, 1 - ACCEPTATION_CIBLE)
            etiq = np.r_[np.ones(len(sc)), np.zeros(len(si))]
            lignes_.append(dict(architecture=arch, inconnue=inconnue, methode=meth,
                                rejet_a_95=round(float((si < seuil).float().mean()), 3),
                                auroc=round(float(roc_auc_score(etiq, np.r_[sc.numpy(), si.numpy()])), 3)))
        del m; torch.cuda.empty_cache()
    return pd.DataFrame(lignes_)

class AvecCaracteristiques(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): f, l = caracteristiques_et_logits(self.m, x); return l, f

def exporter_et_chronometrer(arch, nom):
    m_cpu = creer_modele(arch); m_cpu.load_state_dict(torch.load(f'{SORTIE}/meilleur_{nom}.pt', map_location='cpu'))
    env = AvecCaracteristiques(m_cpu).eval()
    chemin = f'{SORTIE}/maladies_{arch}.onnx'
    options = dict(input_names=['image'], output_names=['logits', 'caracteristiques'], opset_version=17,
                   dynamic_axes={'image': {0: 'lot'}, 'logits': {0: 'lot'}, 'caracteristiques': {0: 'lot'}})
    try:
        torch.onnx.export(env, torch.randn(1, 3, 224, 224), chemin, dynamo=False, **options)
    except TypeError:
        torch.onnx.export(env, torch.randn(1, 3, 224, 224), chemin, **options)
    env.eval()
    s = ort.InferenceSession(chemin, providers=['CPUExecutionProvider'])
    x = next(iter(L_VAL))[0][:8]
    with torch.no_grad(): ref = env(x)[0].numpy()
    ecart = float(np.abs(s.run(None, {'image': x.numpy()})[0] - ref).max())
    x1 = torch.randn(1, 3, 224, 224)
    for _ in range(3): s.run(None, {'image': x1.numpy()})          # échauffement
    debut = time.time()
    for _ in range(10):
        for f in VUES.values(): s.run(None, {'image': f(x1).numpy()})
    return dict(taille_onnx_mo=round(os.path.getsize(chemin) / 1e6, 1), ms_par_photo_4vues=round((time.time() - debut) / 10 * 1000),
                ecart_onnx=ecart)

## 6. Expérience : les trois réseaux (environ 4 heures)

In [ ]:
RESULTATS, SIMUS = [], []
for arch in ARCHITECTURES:
    print(f'\n===== {arch} =====')
    m, h = entrainer(nom_fichier(arch), arch, T_V2, 'moyenne')
    Le, Ye = logits_4vues(m, L_VAL); Lr, Yr = logits_4vues(m, L_VAL_CENTRE)
    be, br = bilan(Le, Ye), bilan(Lr, Yr)
    del m; torch.cuda.empty_cache()
    simu = inconnues_simulees(arch); SIMUS.append(simu)
    pm = simu[simu.methode == 'proba_max']
    r = dict(architecture=arch, epoque_retenue=h['epoque_retenue'], f1_entieres_4vues=be['f1_macro'], f1_recadrages_4vues=br['f1_macro'],
             fausses_alertes_saines=be['fausses_alertes_saines'], rappel_fusariose=be['rappel_fusariose'],
             auroc_inconnues=round(float(pm.auroc.mean()), 3), rejet_inconnues_a_95=round(float(pm.rejet_a_95.mean()), 3),
             auroc_cercosporiose_inconnue=float(pm[pm.inconnue == 'cercosporiose_noire'].auroc.iloc[0]),
             **exporter_et_chronometrer(arch, nom_fichier(arch)))
    RESULTATS.append(r); print(r)
    pd.DataFrame(RESULTATS).to_csv(f'{SORTIE}/comparaison_reseaux.csv', index=False)
pd.concat(SIMUS).to_csv(f'{SORTIE}/inconnues_simulees_par_reseau.csv', index=False)

## 7. Décision selon la règle fixée à l'avance

In [ ]:
TAB = pd.DataFrame(RESULTATS).set_index('architecture')
print(TAB.to_string())
ref = TAB.loc['efficientnet_b0']
conditions = pd.DataFrame({
    'auroc_superieure': TAB.auroc_inconnues > ref.auroc_inconnues,
    'f1_conserve': TAB.f1_entieres_4vues >= ref.f1_entieres_4vues - 0.005,
    'temps_ok': TAB.ms_par_photo_4vues <= 100})
conditions['retenu_possible'] = conditions.all(1) & (TAB.index != 'efficientnet_b0')
print(conditions.to_string())
candidats = TAB[conditions.retenu_possible]
CHOIX_RESEAU = candidats.auroc_inconnues.idxmax() if len(candidats) else 'efficientnet_b0'
print(f'\nRéseau retenu selon la règle : {CHOIX_RESEAU}')
json.dump(dict(versions=VERSIONS, resultats=RESULTATS, conditions=conditions.reset_index().to_dict('records'), choix=CHOIX_RESEAU,
               regle='AUROC des inconnues simulées (proba_max) > B0 ; F1 entières (4 vues) >= B0 - 0,005 ; <= 100 ms par photo (4 vues)'),
          open(f'{SORTIE}/decision_reseau.json', 'w'), indent=2, ensure_ascii=False, default=str)
print(sorted(os.listdir(SORTIE)))